# Road-Aware Pedestrian Detection — Colab

Drive의 가공 데이터 백업(`rapd_backup_*.zip`)으로 학습과 분석을 실행한다.

- 데이터는 매 세션 `/content`에 풀어서 쓴다 (Drive에서 직접 읽으면 매우 느림).
- 학습 결과, 체크포인트, 분석 결과는 모두 Drive(`DRIVE_DIR`)에 저장한다 → 세션이 끊겨도 남는다.
- 런타임 → 런타임 유형 변경 → GPU(L4 또는 A100 권장) 선택 후 위에서부터 실행.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. 설정 — 자기 Drive 구조에 맞게 고친다

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/rapd"          # 결과를 저장할 Drive 폴더
BACKUP_ZIP = f"{DRIVE_DIR}/rapd_backup_20261010.zip"  # 가공 데이터 백업 (풀면 aihub_f/)
BRANCH = "claude/colab-setup"                        # main에 합친 뒤에는 "main"으로
DATA = "/content/data/aihub_f"
RUNS = f"{DRIVE_DIR}/runs"
RESULTS = f"{DRIVE_DIR}/results"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(RUNS, exist_ok=True); os.makedirs(RESULTS, exist_ok=True)

## 2. 코드와 패키지

In [ ]:
%cd /content
!rm -rf road-aware-pedestrian-detection
!git clone -q -b {BRANCH} https://github.com/leoa4238/road-aware-pedestrian-detection.git
%cd /content/road-aware-pedestrian-detection
# Colab에는 PyTorch가 이미 설치돼 있으므로 나머지만 설치한다
!pip install -q ultralytics==8.3.150 pycocotools

## 3. 데이터 풀기 (매 세션, 몇 분)

In [ ]:
import os
if not os.path.exists(f"{DATA}/dataset.yaml"):
    !mkdir -p /content/data && cp "{BACKUP_ZIP}" /content/data/backup.zip
    !cd /content/data && unzip -q backup.zip && rm backup.zip
!ls {DATA}
!for s in train val test; do echo $s $(ls {DATA}/images/$s | wc -l); done
# 백업에 들어 있는 학습 가중치 위치 확인
!find /content/data -name "best.pt" | head

## 4. 베이스라인 학습 (끊기면 이 셀만 다시 실행 → 이어서 학습)

결과: `RUNS/<name>/weights/{last,best}.pt`, test 성능 json은 `RESULTS/`.

In [ ]:
EPOCHS = 50
MODEL = "yolov8s.pt"
!python aihub/train_baseline.py --data {DATA}/dataset.yaml --model {MODEL} --epochs {EPOCHS} \
    --batch 32 --workers 8 --project {RUNS} --results-dir {RESULTS} --resume

## 5. 분석

`BASELINE` 은 4번에서 학습한 가중치 또는 3번 `find`로 찾은 기존 가중치 경로.

In [ ]:
BASELINE = f"{RUNS}/yolov8s_e50_s0/weights/best.pt"
assert os.path.exists(BASELINE), BASELINE

### 5-1. 일반(COCO) 모델 vs 생활도로 학습 모델

COCO 사전학습 모델을 생활도로 학습 없이 그대로 적용해, 학습한 베이스라인과 같은 코드로 비교한다.

In [ ]:
!python aihub/eval_zeroshot.py --data {DATA} --models yolov8s.pt yolo11s.pt {BASELINE} \
    --out {RESULTS}/zeroshot_test.csv

### 5-2. 베이스라인이 놓친 보행자 (크기·발 위치·시간대·날씨·도로 유형·시나리오별)

In [ ]:
!python aihub/analyze_misses.py --data {DATA} --weights {BASELINE} --out {RESULTS}/misses --examples 30

### 5-3. 위험 보행자 정의 비교

`crossing_minus_sidewalk` 가 클수록 (횡단/차도 통행 시나리오와 보도 통행 시나리오를 잘 가르는) 구분력 있는 정의.
예시 이미지에서 주행 경로 사다리꼴이 실제 차로와 맞는지 보고 `--horizon`, `--hw-bottom` 을 조정한다.

In [ ]:
!python aihub/compare_risk_definitions.py --data {DATA} --out {RESULTS}/risk --examples 40

In [ ]:
# 예시 이미지 몇 장 보기
from IPython.display import Image, display
import glob
for p in sorted(glob.glob(f"{RESULTS}/risk/examples/*.jpg"))[:6]:
    display(Image(p, width=800))